In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-12'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[ 0.0631, -0.3333,  0.6196, -0.3140, -1.1508,  0.0816, -0.0783,  0.7042,
         -0.1964,  0.4932, -1.0956,  0.0864]], device='cuda:0')
Scaled actions :  tensor([[ 0.0631, -0.3333,  0.6196, -0.3140, -1.1508,  0.0816, -0.0783,  0.7042,
         -0.1964,  0.4932, -1.0956,  0.0864]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0711e-10,  2.0808e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4362e-08,
          3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04, -3.4389e-08,
         -8.5621e-08, -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,
          7.6633e-07,  7.1809e-07,  1.5422e-06, -7.0859e-03,  1.8227e-02,
         -9.5517e-03, -1.7194e-06, -4.2811e-06, -2.3818e-06, -7.0895e-03,
          1.8232e-02, -9.5524e-03,  3.8316e-05,  6.3118e-02, -3.3332e-01,
          6.1957e-01, -3.1398e-01, -1.1508e+00,  8.1580e-02, -7.8320e-02,
          7.0416e-01, -1.9644e-01,  4.9324e-01, -1.0956e+00,  8.6366e-02]],
       device='cuda:0')
torques: [ 3.99683909e-16 -9.12643102e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.18425700e-16  2.17253195e-16 -6.61016595e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.93165041e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.0213, -0.4133,  0.1727, -0.2547, -1.8872, -0.1760,  0.3333,  0.3103,
         -0.6727,  0.6266, -1.5610, -0.1151]], device='cuda:0')
Scaled actions :  tensor([[-0.0213, -0.4133,  0.1727, -0.2547, -1.8872, -0.1760,  0.3333,  0.3103,
         -0.6727,  0.6266, -1.5610, -0.1151]], device='cuda:0')
obs :  tensor([[ 2.6310e-01, -1.0958e-01, -3.3668e-01, -2.6780e-03, -5.8436e-03,
         -9.9998e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4684e-02,
         -1.0127e-02,  8.6293e-03,  1.3892e-02, -8.7309e-02,  2.2367e-02,
         -2.2678e-02,  6.0820e-04, -1.8399e-02,  3.8273e-02, -1.0820e-01,
          2.4952e-02,  8.0161e-02, -8.4333e-02,  9.5231e-02,  8.8665e-02,
         -7.9455e-01,  9.8651e-02, -1.2934e-01,  2.9970e-03, -1.6061e-01,
          3.2588e-01, -9.7348e-01,  1.2822e-01, -2.1346e-02, -4.1331e-01,
          1.7268e-01, -2.5467e-01, -1.8872e+00, -1.7598e-01,  3.3335e-01,
          3.1025e-01, -6.7275e-01,  6.2657e-01, -1.5610e+00, -1.1

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[-0.6441,  0.9809,  0.8347, -1.2794, -0.2439, -0.7449,  0.6344,  0.6114,
          0.1959, -0.1167,  0.1596,  0.2430]], device='cuda:0')
Scaled actions :  tensor([[-0.6441,  0.9809,  0.8347, -1.2794, -0.2439, -0.7449,  0.6344,  0.6114,
          0.1959, -0.1167,  0.1596,  0.2430]], device='cuda:0')
obs :  tensor([[ 4.4395e-01, -4.6919e-02, -8.7037e-01, -5.9694e-03, -2.0886e-02,
         -9.9976e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.1084e-02,
         -3.0327e-02,  4.0074e-02,  3.1898e-02, -3.3475e-01, -2.7239e-02,
          3.7149e-03,  8.9663e-04, -7.1548e-02,  1.3305e-01, -3.9268e-01,
         -1.3714e-02, -3.6785e-02, -1.2471e-01,  1.9677e-01,  1.0842e-01,
         -1.5828e+00, -2.7442e-01,  3.4827e-01,  1.5533e-03, -3.5709e-01,
          5.8356e-01, -1.7351e+00, -2.2180e-01, -6.4409e-01,  9.8089e-01,
          8.3472e-01, -1.2794e+00, -2.4394e-01, -7.4488e-01,  6.3442e-01,
          6.1144e-01,  1.9592e-01, -1.1669e-01,  1.5957e-01,  2.4

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.4406,  0.8446, -0.3337, -1.4183,  1.3020, -0.0914, -0.1495,  0.0752,
         -0.7468, -0.4987,  0.6382,  0.1384]], device='cuda:0')
Scaled actions :  tensor([[-0.4406,  0.8446, -0.3337, -1.4183,  1.3020, -0.0914, -0.1495,  0.0752,
         -0.7468, -0.4987,  0.6382,  0.1384]], device='cuda:0')
obs :  tensor([[-0.1430, -0.7228, -0.8603, -0.0215, -0.0260, -0.9994,  1.0000,  0.0000,
          0.0000, -0.0487, -0.0308,  0.1039,  0.0443, -0.5544, -0.1772,  0.1227,
          0.0243, -0.1006,  0.2095, -0.6347,  0.0353, -0.5196,  0.0745,  0.4168,
          0.0229, -0.7236, -0.9882,  0.7083,  0.2308,  0.0214,  0.2236, -0.7824,
          0.4363, -0.4406,  0.8446, -0.3337, -1.4183,  1.3020, -0.0914, -0.1495,
          0.0752, -0.7468, -0.4987,  0.6382,  0.1384]], device='cuda:0')
torques: [-200.          200.          200.         -200.          200.
  200.         -200.          200.          200.         -200.
  200.          -13.17180297]
データ収集: step 5


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[ 0.8333, -1.0561, -0.8175, -1.0029,  1.0750,  0.9933, -1.3155, -1.0817,
         -0.7589, -0.5815, -0.1104, -0.6130]], device='cuda:0')
Scaled actions :  tensor([[ 0.8333, -1.0561, -0.8175, -1.0029,  1.0750,  0.9933, -1.3155, -1.0817,
         -0.7589, -0.5815, -0.1104, -0.6130]], device='cuda:0')
obs :  tensor([[-0.4044, -0.0178, -0.4116, -0.0348, -0.0145, -0.9993,  1.0000,  0.0000,
          0.0000, -0.1764, -0.0106,  0.1609,  0.0402, -0.5938, -0.2734,  0.2098,
          0.0829, -0.1220,  0.2339, -0.6836,  0.1560, -0.6050,  0.1296,  0.1801,
         -0.0487,  0.2325, -0.0528,  0.2263,  0.3210, -0.2108,  0.0404,  0.1963,
          0.4376,  0.8333, -1.0561, -0.8175, -1.0029,  1.0750,  0.9933, -1.3155,
         -1.0817, -0.7589, -0.5815, -0.1104, -0.6130]], device='cuda:0')
torques: [  62.85144849  200.         -200.         -200.          200.
  200.         -200.         -200.         -200.         -200.
  200.         -200.        ]
データ収集: step 6


In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.5440,  0.4894,  1.0696, -0.8569,  0.1385,  0.9336, -0.2183,  0.1111,
          1.0381,  0.2992, -0.5799, -0.5461]], device='cuda:0')
Scaled actions :  tensor([[ 0.5440,  0.4894,  1.0696, -0.8569,  0.1385,  0.9336, -0.2183,  0.1111,
          1.0381,  0.2992, -0.5799, -0.5461]], device='cuda:0')
obs :  tensor([[ 0.2108,  0.6640, -0.4688, -0.0203, -0.0122, -0.9997,  1.0000,  0.0000,
          0.0000, -0.2448, -0.0092,  0.1708,  0.0174, -0.4418, -0.1740,  0.2111,
          0.1216, -0.1825,  0.2295, -0.5453,  0.1374, -0.1294, -0.0799, -0.0588,
         -0.1851,  1.1996,  0.9479, -0.1668,  0.0957, -0.3679, -0.0734,  0.9252,
         -0.5268,  0.5440,  0.4894,  1.0696, -0.8569,  0.1385,  0.9336, -0.2183,
          0.1111,  1.0381,  0.2992, -0.5799, -0.5461]], device='cuda:0')
torques: [ 200.         -200.         -200.         -200.          200.
  200.         -200.         -200.         -200.         -200.
  -32.75561925 -200.        ]
データ収集: step 7


In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.0676,  0.6563,  1.0318,  0.4012,  0.3726,  0.6053, -0.1122, -0.0459,
          0.9241,  0.8364, -0.6936,  0.3730]], device='cuda:0')
Scaled actions :  tensor([[ 0.0676,  0.6563,  1.0318,  0.4012,  0.3726,  0.6053, -0.1122, -0.0459,
          0.9241,  0.8364, -0.6936,  0.3730]], device='cuda:0')
obs :  tensor([[ 0.3029,  0.0023, -0.4985, -0.0064, -0.0225, -0.9997,  1.0000,  0.0000,
          0.0000, -0.2043, -0.0249,  0.1826, -0.0266, -0.2154,  0.0769,  0.1299,
          0.1515, -0.2228,  0.2234, -0.4680, -0.0551,  0.5349, -0.1042,  0.0552,
         -0.0285,  1.0033,  1.2253, -0.5145,  0.1639, -0.0507,  0.0031, -0.0555,
         -1.0581,  0.0676,  0.6563,  1.0318,  0.4012,  0.3726,  0.6053, -0.1122,
         -0.0459,  0.9241,  0.8364, -0.6936,  0.3730]], device='cuda:0')
torques: [ 200.          200.          200.         -200.         -200.
 -200.          200.         -200.          200.          153.45744923
 -200.           72.56339475]
データ収集:

In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[-0.7539,  1.1181, -0.7818, -1.8143, -0.3786,  0.0377,  0.8326, -0.2243,
         -0.6318,  0.0765,  0.2748,  0.2047]], device='cuda:0')
Scaled actions :  tensor([[-0.7539,  1.1181, -0.7818, -1.8143, -0.3786,  0.0377,  0.8326, -0.2243,
         -0.6318,  0.0765,  0.2748,  0.2047]], device='cuda:0')
obs :  tensor([[ 0.3498, -0.6327, -0.3823, -0.0195, -0.0365, -0.9991,  1.0000,  0.0000,
          0.0000, -0.0936, -0.0574,  0.1886,  0.0183, -0.0718,  0.2669,  0.0298,
          0.1711, -0.2095,  0.2390, -0.5442, -0.1565,  0.4168, -0.1791,  0.0511,
          0.3723,  0.7380,  0.7060, -0.3761,  0.0230,  0.1397,  0.1407, -0.3462,
         -0.0524, -0.7539,  1.1181, -0.7818, -1.8143, -0.3786,  0.0377,  0.8326,
         -0.2243, -0.6318,  0.0765,  0.2748,  0.2047]], device='cuda:0')
torques: [ -94.71679348  200.          200.          200.          200.
  -26.91073659  101.38543705 -200.          200.          200.
   62.50598823  200.        ]
データ収集: step 9


In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.2797,  0.7846, -0.3101, -2.1573, -0.7804, -0.2842,  0.4166,  0.3500,
         -0.3726, -0.3458,  0.2208, -0.0359]], device='cuda:0')
Scaled actions :  tensor([[-0.2797,  0.7846, -0.3101, -2.1573, -0.7804, -0.2842,  0.4166,  0.3500,
         -0.3726, -0.3458,  0.2208, -0.0359]], device='cuda:0')
obs :  tensor([[ 0.2988,  0.0794, -0.3170, -0.0285, -0.0498, -0.9984,  1.0000,  0.0000,
          0.0000, -0.0641, -0.0842,  0.1763,  0.0783, -0.0284,  0.2973,  0.0214,
          0.1532, -0.2166,  0.2488, -0.5054, -0.0746, -0.0712, -0.1021, -0.1623,
          0.2450, -0.2090, -0.3021,  0.2283, -0.1729, -0.1753, -0.0279,  0.6359,
          0.6008, -0.2797,  0.7846, -0.3101, -2.1573, -0.7804, -0.2842,  0.4166,
          0.3500, -0.3726, -0.3458,  0.2208, -0.0359]], device='cuda:0')
torques: [-200.          200.         -200.         -200.         -200.
 -200.          200.         -200.         -200.         -200.
  200.          -28.31459898]
データ収集: step 10

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[ 0.3638, -0.0518,  1.2389, -0.9868,  0.2973,  0.5930, -0.7476,  0.9322,
          0.8726,  1.1554, -1.1495,  0.2488]], device='cuda:0')
Scaled actions :  tensor([[ 0.3638, -0.0518,  1.2389, -0.9868,  0.2973,  0.5930, -0.7476,  0.9322,
          0.8726,  1.1554, -1.1495,  0.2488]], device='cuda:0')
obs :  tensor([[-0.3397,  0.5432, -0.3809, -0.0135, -0.0480, -0.9988,  1.0000,  0.0000,
          0.0000, -0.1168, -0.0748,  0.1256,  0.1181, -0.1745,  0.1429,  0.1111,
          0.1392, -0.2642,  0.2136, -0.2952, -0.0408, -0.3326,  0.1794, -0.3027,
          0.1568, -1.1552, -0.9216,  0.5779,  0.0073, -0.2530, -0.3080,  1.1190,
          0.0117,  0.3638, -0.0518,  1.2389, -0.9868,  0.2973,  0.5930, -0.7476,
          0.9322,  0.8726,  1.1554, -1.1495,  0.2488]], device='cuda:0')
torques: [  -5.84766249  200.         -200.         -200.         -195.45254756
   57.06716261   64.13084341  200.           29.03250096 -200.
  -80.93455828   -6.36239268]
データ収集

In [50]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 272
Original actions :  tensor([[ 0.0195, -0.0470,  0.4485,  0.3934,  0.1298,  0.1871, -0.5771, -1.1013,
          1.8468, -0.0021,  1.5698, -1.8502]], device='cuda:0')
Scaled actions :  tensor([[ 0.0195, -0.0470,  0.4485,  0.3934,  0.1298,  0.1871, -0.5771, -1.1013,
          1.8468, -0.0021,  1.5698, -1.8502]], device='cuda:0')
obs :  tensor([[-0.4623,  1.1451, -0.2924,  0.0101,  0.1958, -0.9806,  1.0000,  0.0000,
          0.0000, -0.2827, -0.2245,  0.4114,  0.1073, -0.1396,  0.2529, -0.0542,
         -0.0713,  0.0658, -0.3659, -0.4650,  0.6453,  0.0261,  0.2726, -0.6345,
          0.6084,  0.5415,  1.0223,  0.7865, -0.0163,  0.0784, -0.9732, -0.0501,
         -0.1860,  0.0195, -0.0470,  0.4485,  0.3934,  0.1298,  0.1871, -0.5771,
         -1.1013,  1.8468, -0.0021,  1.5698, -1.8502]], device='cuda:0')
torques: [-200.          200.         -200.          200.          -28.99366316
  -82.28326734  200.         -200.         -200.         -200.
  200.         -200.        ]
データ収

In [54]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=0.970, Scaled action max=0.970
Step 1/10, Total steps: 303
steps: 303
actions : tensor([[ 0.0012, -0.6184,  0.6507,  0.0122, -0.2181, -0.1777, -0.0656,  0.9697,
         -0.3566,  0.3218, -1.1195,  0.5678]], device='cuda:0')
target_dof_pos: tensor([[-0.0664, -0.3396, -0.9487,  1.1152, -0.0904,  0.7073, -0.4281, -0.3809,
         -1.9965,  2.0266, -1.1171, -0.2107]], device='cuda:0')
Step 1: Original action max=1.273, Scaled action max=1.273
Step 2: Original action max=1.486, Scaled action max=1.486
データ収集完了: 10 steps collected with action_scale=1.0


In [97]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [55]:
env.sim.stop()

In [99]:
env.reset()
cnt = 0

In [56]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-8_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (313, 58)
